# Notebook 50 — Out-of-Family Extrapolation and Regime Interpolation

This notebook stress-tests the meta-law beyond leave-one-regime-out evaluation.

We evaluate:

- extrapolation in `k`
- leave-one-forcing-family-out
- leave-one-system-out
- leave-one-task-out
- missing-point interpolation
- mixed metadata extrapolation

The goal is to determine whether the metadata-conditioned law is merely an in-family interpolator,
or a more general structural law over regime space.

## Fixed governing template

\[
g(r,c;\beta)=\beta_0+\beta_1 c+\beta_2 r+\beta_3 c^3+\beta_4 r^2+\beta_5 r c^2
\]

## Main questions

1. Can the meta-law extrapolate to unseen `k` values?
2. Can it transfer to unseen forcing families, systems, or tasks?
3. Can it interpolate missing points in regime space?
4. Where does linear metadata-to-coefficient prediction begin to fail?

In [ ]:
# Core imports

import warnings
warnings.filterwarnings("ignore")

import os
import glob
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neural_network import MLPRegressor

try:
    from IPython.display import display
except Exception:
    display = print

np.random.seed(42)

In [ ]:
# Data discovery and synthetic fallback

DATA_PATH = None

def autodetect_data_path():
    candidates = []
    patterns = [
        "*residual*flow*.parquet",
        "*residual*flow*.csv",
        "*governing*flow*.parquet",
        "*governing*flow*.csv",
        "*.parquet",
        "*.csv",
        "*.pkl",
        "*.pickle",
    ]
    for pat in patterns:
        candidates.extend(glob.glob(pat))
        candidates.extend(glob.glob(os.path.join("data", pat)))
        candidates.extend(glob.glob(os.path.join("outputs", pat)))
    candidates = [c for c in candidates if os.path.isfile(c)]
    return candidates[0] if candidates else None

def synthetic_dataset():
    systems = ["entropy", "unevenness"]
    tasks = ["zeta_vs_gue", "zeta_vs_poisson"]
    forcing_modes = ["capacity_gap", "feature_gap", "condition_gap"]
    ks = [3, 5, 7]
    flow_modes = ["linear", "nonlinear"]

    rows = []
    sample_id = 0
    for system in systems:
        for task in tasks:
            for forcing_mode in forcing_modes:
                for k in ks:
                    for flow_mode in flow_modes:
                        n_paths = 14
                        n_steps = 42
                        c_grid = np.linspace(-1.25, 1.05, n_steps)

                        sys_shift = 0.06 if system == "entropy" else -0.04
                        task_shift = 0.05 if task == "zeta_vs_gue" else -0.03
                        force_shift = {"capacity_gap": 0.00, "feature_gap": 0.03, "condition_gap": 0.08}[forcing_mode]
                        k_shift = {3: -0.05, 5: 0.02, 7: 0.06}[k]
                        flow_shift = 0.05 if flow_mode == "nonlinear" else -0.02
                        nl_gain = 1.0 if flow_mode == "nonlinear" else 0.55

                        for path_id in range(n_paths):
                            r = -0.75 + 0.10 * path_id + 0.05 * np.sin(0.7 * path_id)
                            for window_id, c in enumerate(c_grid):
                                g = (
                                    0.58 * np.tanh(1.35 * c)
                                    + 0.42 * c
                                    - 0.78 * r
                                    + 0.20 * r**2
                                    + nl_gain * 0.07 * c**2
                                    + nl_gain * 0.10 * r * c
                                    - nl_gain * 0.025 * r**3
                                    + sys_shift + task_shift + force_shift + k_shift + flow_shift
                                )
                                if forcing_mode == "condition_gap":
                                    g += 0.06 * np.sin(2.3 * c)
                                if system == "entropy":
                                    g += 0.03 * np.cos(1.2 * c)
                                if task == "zeta_vs_poisson":
                                    g -= 0.015 * c
                                if flow_mode == "linear":
                                    g -= 0.09 * r**2
                                    g += 0.015 * c * r

                                delta_c = c_grid[min(window_id + 1, n_steps - 1)] - c if window_id < n_steps - 1 else c - c_grid[max(window_id - 1, 0)]
                                noise = 0.012 * np.random.randn()
                                next_r = r + (g + noise) * delta_c

                                rows.append(
                                    {
                                        "system": system,
                                        "task": task,
                                        "forcing_mode": forcing_mode,
                                        "k": k,
                                        "flow_mode": flow_mode,
                                        "condition_coord": c,
                                        "residual": r,
                                        "predicted_flow": g + noise,
                                        "next_residual": next_r,
                                        "delta_condition": delta_c,
                                        "sample_id": sample_id,
                                        "path_id": path_id,
                                        "window_id": window_id,
                                    }
                                )
                                r = next_r
                                sample_id += 1
    return pd.DataFrame(rows)

if DATA_PATH is None:
    DATA_PATH = autodetect_data_path()

USE_SYNTHETIC_FALLBACK = DATA_PATH is None
print("Selected DATA_PATH:", DATA_PATH)
print("USE_SYNTHETIC_FALLBACK:", USE_SYNTHETIC_FALLBACK)

In [ ]:
# Loading + schema alignment

def load_dataframe(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".parquet":
        return pd.read_parquet(path)
    if ext in [".pkl", ".pickle"]:
        return pd.read_pickle(path)
    return pd.read_csv(path)

alias_groups = {
    "condition_coord": ["condition_coord", "c", "condition", "cond", "condition_coordinate"],
    "residual": ["residual", "r", "resid"],
    "predicted_flow": ["predicted_flow", "flow", "g", "drdc", "delta_residual_per_condition"],
    "next_residual": ["next_residual", "r_next", "next_r"],
    "delta_condition": ["delta_condition", "dc", "delta_c"],
    "forcing_mode": ["forcing_mode", "forcing", "forcing_gap", "mode"],
}

def align_schema(df):
    cols = list(df.columns)
    rename_map = {}
    for canonical, aliases in alias_groups.items():
        for a in aliases:
            if a in cols and a != canonical:
                rename_map[a] = canonical
                break
    df = df.rename(columns=rename_map)

    if "next_residual" not in df.columns and {"residual", "predicted_flow", "delta_condition"}.issubset(df.columns):
        df["next_residual"] = df["residual"] + df["predicted_flow"] * df["delta_condition"]

    if "delta_condition" not in df.columns and "condition_coord" in df.columns:
        df = df.sort_values(["condition_coord"]).copy()
        dc = np.gradient(df["condition_coord"].to_numpy())
        df["delta_condition"] = dc

    defaults = {
        "system": "unknown_system",
        "task": "unknown_task",
        "forcing_mode": "unknown_forcing",
        "k": 5,
        "flow_mode": "unknown_flow",
        "sample_id": np.arange(len(df)),
        "path_id": 0,
        "window_id": np.arange(len(df)),
    }
    for k, v in defaults.items():
        if k not in df.columns:
            df[k] = v

    required = ["condition_coord", "residual", "predicted_flow"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns after alignment: {missing}")
    return df

if USE_SYNTHETIC_FALLBACK:
    df = synthetic_dataset()
    data_source = "synthetic_fallback"
else:
    df = align_schema(load_dataframe(DATA_PATH))
    data_source = DATA_PATH

df = align_schema(df)
print("Data source:", data_source)
print("Shape:", df.shape)
display(df.head())

In [ ]:
# Fixed template and per-regime coefficient dataset

TERM_NAMES = ["1", "c", "r", "c^3", "r^2", "r c^2"]
COEF_COLS = [f"coef_{t}" for t in TERM_NAMES]

def design_template(data):
    r = data["residual"].to_numpy(dtype=float)
    c = data["condition_coord"].to_numpy(dtype=float)
    return np.column_stack([
        np.ones_like(r),
        c,
        r,
        c**3,
        r**2,
        r * c**2,
    ])

def fit_template(sub, alpha=1e-6):
    X = design_template(sub)
    y = sub["predicted_flow"].to_numpy(dtype=float)
    beta = np.linalg.solve(X.T @ X + alpha * np.eye(X.shape[1]), X.T @ y)
    pred = X @ beta
    stats = {"n": len(sub), "rmse": math.sqrt(mean_squared_error(y, pred))}
    for name, coef in zip(TERM_NAMES, beta):
        stats[f"coef_{name}"] = float(coef)
    return beta, pred, stats

def predict_with_beta(sub, beta):
    return design_template(sub) @ beta

def safe_corr(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])

def explained_variance_score_manual(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.var(y_true)
    if denom < 1e-12:
        return np.nan
    return float(1.0 - np.var(y_true - y_pred) / denom)

def trajectory_gap(sub, beta_ref, beta_cmp, n_r0=15):
    cmin, cmax = sub["condition_coord"].min(), sub["condition_coord"].max()
    rmin, rmax = sub["residual"].min(), sub["residual"].max()
    cgrid = np.linspace(cmin, cmax, 40)
    r0s = np.linspace(np.quantile(sub["residual"], 0.05), np.quantile(sub["residual"], 0.95), n_r0)
    flow_cap = max(1.0, 2.5 * np.quantile(np.abs(sub["predicted_flow"]), 0.995))

    def integrate(beta, r0):
        r = float(np.clip(r0, rmin, rmax))
        traj = [r]
        for i in range(len(cgrid) - 1):
            c = cgrid[i]
            dc = float(cgrid[i+1] - cgrid[i])
            x = np.array([1.0, c, r, c**3, r**2, r * c**2], dtype=float)
            g = float(np.clip(x @ beta, -flow_cap, flow_cap))
            r = float(np.clip(r + g * dc, rmin - 0.5, rmax + 0.5))
            traj.append(r)
        return np.array(traj)

    errs = []
    for r0 in r0s:
        t_ref = integrate(beta_ref, r0)
        t_cmp = integrate(beta_cmp, r0)
        errs.append(math.sqrt(mean_squared_error(t_ref, t_cmp)))
    return float(np.mean(errs))

regime_rows = []
regime_subsets = {}

for vals, sub in df.groupby(["system", "task", "forcing_mode", "k", "flow_mode"]):
    if len(sub) < 30:
        continue
    beta, pred, stats = fit_template(sub.copy())
    kval = int(vals[3]) if float(vals[3]).is_integer() else vals[3]
    regime_id = f"{vals[0]}|{vals[1]}|{vals[2]}|k={kval}|{vals[4]}"
    row = {
        "regime_id": regime_id,
        "system": vals[0],
        "task": vals[1],
        "forcing_mode": vals[2],
        "k": float(vals[3]),
        "flow_mode": vals[4],
    }
    row.update(stats)
    regime_rows.append(row)
    regime_subsets[regime_id] = sub.copy()

coef_df = pd.DataFrame(regime_rows).reset_index(drop=True)
display(coef_df.head())

In [ ]:
# Metadata matrix

meta_df = coef_df[["regime_id", "system", "task", "forcing_mode", "flow_mode", "k"]].copy()
X_base = pd.get_dummies(meta_df[["system", "task", "forcing_mode", "flow_mode"]], drop_first=False)
X_base["k"] = coef_df["k"].astype(float).values
X_base["k2"] = coef_df["k"].astype(float).values**2
Y_coef = coef_df[COEF_COLS].copy()

display(X_base.head())
display(Y_coef.head())

In [ ]:
# Meta-model builders

class MultiTargetMLP:
    def __init__(self, hidden_layer_sizes=(16,), random_state=42, max_iter=4000):
        self.hidden_layer_sizes = hidden_layer_sizes
        self.random_state = random_state
        self.max_iter = max_iter
        self.models = None

    def fit(self, X, Y):
        Y = np.asarray(Y, dtype=float)
        if Y.ndim == 1:
            Y = Y[:, None]
        self.models = [
            MLPRegressor(hidden_layer_sizes=self.hidden_layer_sizes, random_state=self.random_state, max_iter=self.max_iter)
            for _ in range(Y.shape[1])
        ]
        for j, m in enumerate(self.models):
            m.fit(X, Y[:, j])
        return self

    def predict(self, X):
        cols = [m.predict(X) for m in self.models]
        return np.column_stack(cols)

def build_linear():
    return LinearRegression()

def build_ridge():
    return Ridge(alpha=1.0)

def build_mlp():
    return MultiTargetMLP()

def fit_predict_model(X_train, Y_train, X_test, model_name):
    builders = {
        "linear": build_linear,
        "ridge": build_ridge,
        "mlp_small": build_mlp,
    }
    return builders[model_name]().fit(np.asarray(X_train, float), np.asarray(Y_train, float)).predict(np.asarray(X_test, float))

def fit_predict_latent(X_train, Y_train, X_test, model_name):
    coef_scaler = StandardScaler()
    Y_train_std = coef_scaler.fit_transform(np.asarray(Y_train, float))
    pca = PCA(n_components=2, random_state=42)
    Z_train = pca.fit_transform(Y_train_std)
    zhat = fit_predict_model(X_train, Z_train, X_test, model_name)
    yhat_std = pca.inverse_transform(np.atleast_2d(zhat))
    yhat = coef_scaler.inverse_transform(yhat_std)
    return yhat

## Evaluation helper

Each split returns:
- coefficient RMSE
- field RMSE / correlation / explained variance
- trajectory RMSE

for:
- shared-law
- best meta-law under the split
- regime-specific baseline

In [ ]:
# Split evaluation helper

beta_shared = Y_coef.to_numpy(dtype=float).mean(axis=0)

def evaluate_holdout(test_mask, split_name):
    train_mask = ~test_mask

    X_train = X_base.loc[train_mask]
    X_test = X_base.loc[test_mask]
    Y_train = Y_coef.loc[train_mask]
    Y_test = Y_coef.loc[test_mask]
    test_regimes = coef_df.loc[test_mask, "regime_id"].tolist()

    rows = []
    for i, regime_id in enumerate(test_regimes):
        x_te = X_test.iloc[[i]]
        beta_true = Y_test.iloc[i].to_numpy(dtype=float)

        candidates = {
            "linear": fit_predict_model(X_train, Y_train, x_te, "linear")[0],
            "ridge": fit_predict_model(X_train, Y_train, x_te, "ridge")[0],
            "mlp_small": fit_predict_model(X_train, Y_train, x_te, "mlp_small")[0],
            "latent_linear": fit_predict_latent(X_train, Y_train, x_te, "linear")[0],
            "latent_ridge": fit_predict_latent(X_train, Y_train, x_te, "ridge")[0],
            "latent_mlp_small": fit_predict_latent(X_train, Y_train, x_te, "mlp_small")[0],
        }

        best_name = min(candidates, key=lambda k: math.sqrt(mean_squared_error(beta_true, candidates[k])))
        beta_meta = candidates[best_name]

        sub = regime_subsets[regime_id]
        y_emp = sub["predicted_flow"].to_numpy(dtype=float)
        pred_shared = predict_with_beta(sub, beta_shared)
        pred_meta = predict_with_beta(sub, beta_meta)
        pred_specific = predict_with_beta(sub, beta_true)

        rows.append({
            "split_name": split_name,
            "regime_id": regime_id,
            "best_meta_model": best_name,

            "coef_rmse_shared": math.sqrt(mean_squared_error(beta_true, beta_shared)),
            "coef_rmse_meta": math.sqrt(mean_squared_error(beta_true, beta_meta)),

            "field_rmse_shared": math.sqrt(mean_squared_error(y_emp, pred_shared)),
            "field_rmse_meta": math.sqrt(mean_squared_error(y_emp, pred_meta)),
            "field_rmse_specific": math.sqrt(mean_squared_error(y_emp, pred_specific)),

            "field_corr_shared": safe_corr(y_emp, pred_shared),
            "field_corr_meta": safe_corr(y_emp, pred_meta),
            "field_corr_specific": safe_corr(y_emp, pred_specific),

            "field_ev_shared": explained_variance_score_manual(y_emp, pred_shared),
            "field_ev_meta": explained_variance_score_manual(y_emp, pred_meta),
            "field_ev_specific": explained_variance_score_manual(y_emp, pred_specific),

            "traj_rmse_shared": trajectory_gap(sub, beta_true, beta_shared),
            "traj_rmse_meta": trajectory_gap(sub, beta_true, beta_meta),
        })
    return pd.DataFrame(rows)

## Experiment A — Extrapolation in k

In [ ]:
# k extrapolation / interpolation splits

k_results = []

for train_ks, test_k, label in [
    ([3, 5], 7, "k_extrapolate_high"),
    ([5, 7], 3, "k_extrapolate_low"),
    ([3, 7], 5, "k_interpolate_mid"),
]:
    test_mask = coef_df["k"].astype(float) == float(test_k)
    res = evaluate_holdout(test_mask, label)
    res["train_ks"] = str(train_ks)
    res["test_k"] = test_k
    k_results.append(res)

k_holdout_df = pd.concat(k_results, ignore_index=True)
display(k_holdout_df.head())

## Experiment B — Leave-one-forcing-family-out

In [ ]:
forcing_results = []
for fmode in sorted(coef_df["forcing_mode"].astype(str).unique()):
    test_mask = coef_df["forcing_mode"].astype(str) == fmode
    forcing_results.append(evaluate_holdout(test_mask, f"forcing_holdout::{fmode}"))

forcing_holdout_df = pd.concat(forcing_results, ignore_index=True)
display(forcing_holdout_df.head())

## Experiment C — Leave-one-system-out

In [ ]:
system_results = []
for system in sorted(coef_df["system"].astype(str).unique()):
    test_mask = coef_df["system"].astype(str) == system
    system_results.append(evaluate_holdout(test_mask, f"system_holdout::{system}"))

system_holdout_df = pd.concat(system_results, ignore_index=True)
display(system_holdout_df.head())

## Experiment D — Leave-one-task-out

In [ ]:
task_results = []
for task in sorted(coef_df["task"].astype(str).unique()):
    test_mask = coef_df["task"].astype(str) == task
    task_results.append(evaluate_holdout(test_mask, f"task_holdout::{task}"))

task_holdout_df = pd.concat(task_results, ignore_index=True)
display(task_holdout_df.head())

## Experiment E — Missing-point interpolation

In [ ]:
# missing-point interpolation: remove one interior regime at a time (focus on k=5 where available)

interp_results = []
interp_mask = coef_df["k"].astype(float) == 5.0
for regime_id in coef_df.loc[interp_mask, "regime_id"].tolist():
    test_mask = coef_df["regime_id"] == regime_id
    interp_results.append(evaluate_holdout(test_mask, f"missing_point::{regime_id}"))

interp_holdout_df = pd.concat(interp_results, ignore_index=True)
display(interp_holdout_df.head())

## Experiment F — Mixed metadata extrapolation

In [ ]:
# mixed extrapolation: hold out all regimes in one system + one forcing mode combination

mixed_results = []
for system in sorted(coef_df["system"].astype(str).unique()):
    for fmode in sorted(coef_df["forcing_mode"].astype(str).unique()):
        test_mask = (
            (coef_df["system"].astype(str) == system) &
            (coef_df["forcing_mode"].astype(str) == fmode)
        )
        if test_mask.sum() == 0 or test_mask.sum() == len(coef_df):
            continue
        mixed_results.append(evaluate_holdout(test_mask, f"mixed::{system}|{fmode}"))

mixed_holdout_df = pd.concat(mixed_results, ignore_index=True)
display(mixed_holdout_df.head())

## Summary tables

In [ ]:
def summarize_block(df_block, name):
    return pd.DataFrame([{
        "block": name,
        "coef_rmse_shared": df_block["coef_rmse_shared"].mean(),
        "coef_rmse_meta": df_block["coef_rmse_meta"].mean(),
        "field_rmse_shared": df_block["field_rmse_shared"].mean(),
        "field_rmse_meta": df_block["field_rmse_meta"].mean(),
        "traj_rmse_shared": df_block["traj_rmse_shared"].mean(),
        "traj_rmse_meta": df_block["traj_rmse_meta"].mean(),
    }])

summary_df = pd.concat([
    summarize_block(k_holdout_df, "k_extrapolation_interpolation"),
    summarize_block(forcing_holdout_df, "forcing_family_holdout"),
    summarize_block(system_holdout_df, "system_holdout"),
    summarize_block(task_holdout_df, "task_holdout"),
    summarize_block(interp_holdout_df, "missing_point_interpolation"),
    summarize_block(mixed_holdout_df, "mixed_metadata_extrapolation"),
], ignore_index=True)

display(summary_df)

In [ ]:
# Plot summary by block

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].bar(summary_df["block"], summary_df["coef_rmse_meta"])
axes[0].set_title("Meta-law coefficient RMSE")
axes[0].tick_params(axis="x", rotation=35)

axes[1].bar(summary_df["block"], summary_df["field_rmse_meta"])
axes[1].set_title("Meta-law field RMSE")
axes[1].tick_params(axis="x", rotation=35)

axes[2].bar(summary_df["block"], summary_df["traj_rmse_meta"])
axes[2].set_title("Meta-law trajectory RMSE")
axes[2].tick_params(axis="x", rotation=35)

plt.tight_layout()
plt.show()

In [ ]:
# Shared vs meta comparison by block

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

x = np.arange(len(summary_df))
w = 0.38

axes[0].bar(x - w/2, summary_df["coef_rmse_shared"], width=w, label="shared")
axes[0].bar(x + w/2, summary_df["coef_rmse_meta"], width=w, label="meta")
axes[0].set_title("Coefficient RMSE")
axes[0].set_xticks(x)
axes[0].set_xticklabels(summary_df["block"], rotation=35, ha="right")
axes[0].legend()

axes[1].bar(x - w/2, summary_df["field_rmse_shared"], width=w, label="shared")
axes[1].bar(x + w/2, summary_df["field_rmse_meta"], width=w, label="meta")
axes[1].set_title("Field RMSE")
axes[1].set_xticks(x)
axes[1].set_xticklabels(summary_df["block"], rotation=35, ha="right")
axes[1].legend()

axes[2].bar(x - w/2, summary_df["traj_rmse_shared"], width=w, label="shared")
axes[2].bar(x + w/2, summary_df["traj_rmse_meta"], width=w, label="meta")
axes[2].set_title("Trajectory RMSE")
axes[2].set_xticks(x)
axes[2].set_xticklabels(summary_df["block"], rotation=35, ha="right")
axes[2].legend()

plt.tight_layout()
plt.show()

## Error vs distance from training support (k tests)

In [ ]:
# Error vs distance for k tests

k_dist_rows = []
for _, row in k_holdout_df.iterrows():
    split_name = row["split_name"]
    if "high" in split_name:
        dist = 2
    elif "low" in split_name:
        dist = 2
    else:
        dist = 0
    k_dist_rows.append({
        "split_name": split_name,
        "distance_from_training_support": dist,
        "coef_rmse_meta": row["coef_rmse_meta"],
        "traj_rmse_meta": row["traj_rmse_meta"],
    })
k_dist_df = pd.DataFrame(k_dist_rows)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(k_dist_df["distance_from_training_support"], k_dist_df["coef_rmse_meta"])
axes[0].set_xlabel("distance from training support")
axes[0].set_ylabel("meta coefficient RMSE")
axes[0].set_title("k-test coefficient error")

axes[1].scatter(k_dist_df["distance_from_training_support"], k_dist_df["traj_rmse_meta"])
axes[1].set_xlabel("distance from training support")
axes[1].set_ylabel("meta trajectory RMSE")
axes[1].set_title("k-test trajectory error")

plt.tight_layout()
plt.show()

## Best meta-model winners under harder tests

In [ ]:
winner_counts_df = pd.concat([
    k_holdout_df[["best_meta_model"]].assign(block="k"),
    forcing_holdout_df[["best_meta_model"]].assign(block="forcing"),
    system_holdout_df[["best_meta_model"]].assign(block="system"),
    task_holdout_df[["best_meta_model"]].assign(block="task"),
    interp_holdout_df[["best_meta_model"]].assign(block="interp"),
    mixed_holdout_df[["best_meta_model"]].assign(block="mixed"),
], ignore_index=True)

winner_summary = winner_counts_df.groupby(["block", "best_meta_model"]).size().reset_index(name="count")
display(winner_summary)

pivot = winner_summary.pivot(index="best_meta_model", columns="block", values="count").fillna(0)
pivot.plot(kind="bar", figsize=(10, 5))
plt.title("Best meta-model winner counts by evaluation block")
plt.ylabel("count")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## Representative success and failure cases

In [ ]:
# Pick one success case and one harder case

all_blocks = pd.concat([
    k_holdout_df.assign(block="k"),
    forcing_holdout_df.assign(block="forcing"),
    system_holdout_df.assign(block="system"),
    task_holdout_df.assign(block="task"),
    interp_holdout_df.assign(block="interp"),
    mixed_holdout_df.assign(block="mixed"),
], ignore_index=True)

success_row = all_blocks.iloc[int(np.argmin(all_blocks["traj_rmse_meta"].to_numpy()))]
failure_row = all_blocks.iloc[int(np.argmax(all_blocks["traj_rmse_meta"].to_numpy()))]

display(success_row.to_frame().T)
display(failure_row.to_frame().T)

In [ ]:
# Plot trajectories for representative success and harder case

def reconstruct_best_beta_for_regime(regime_id):
    train_mask = coef_df["regime_id"] != regime_id
    test_mask = coef_df["regime_id"] == regime_id
    X_train = X_base.loc[train_mask]
    X_test = X_base.loc[test_mask]
    Y_train = Y_coef.loc[train_mask]
    Y_test = Y_coef.loc[test_mask]
    beta_true = Y_test.to_numpy(dtype=float)[0]

    candidates = {
        "linear": fit_predict_model(X_train, Y_train, X_test, "linear")[0],
        "ridge": fit_predict_model(X_train, Y_train, X_test, "ridge")[0],
        "mlp_small": fit_predict_model(X_train, Y_train, X_test, "mlp_small")[0],
        "latent_linear": fit_predict_latent(X_train, Y_train, X_test, "linear")[0],
        "latent_ridge": fit_predict_latent(X_train, Y_train, X_test, "ridge")[0],
        "latent_mlp_small": fit_predict_latent(X_train, Y_train, X_test, "mlp_small")[0],
    }
    best_name = min(candidates, key=lambda k: math.sqrt(mean_squared_error(beta_true, candidates[k])))
    return beta_true, candidates[best_name], best_name

def plot_case(regime_id, title_prefix):
    beta_true, beta_meta, best_name = reconstruct_best_beta_for_regime(regime_id)
    sub = regime_subsets[regime_id]

    cmin, cmax = sub["condition_coord"].min(), sub["condition_coord"].max()
    rmin, rmax = sub["residual"].min(), sub["residual"].max()
    cgrid = np.linspace(cmin, cmax, 45)
    r0s = np.linspace(np.quantile(sub["residual"], 0.1), np.quantile(sub["residual"], 0.9), 8)
    flow_cap = max(1.0, 2.5 * np.quantile(np.abs(sub["predicted_flow"]), 0.995))

    def integrate_beta(beta, r0):
        r = float(np.clip(r0, rmin, rmax))
        traj = [r]
        for j in range(len(cgrid) - 1):
            c = cgrid[j]
            dc = float(cgrid[j+1] - cgrid[j])
            x = np.array([1.0, c, r, c**3, r**2, r * c**2], dtype=float)
            g = float(np.clip(x @ beta, -flow_cap, flow_cap))
            r = float(np.clip(r + g * dc, rmin - 0.5, rmax + 0.5))
            traj.append(r)
        return np.array(traj)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    for r0 in r0s:
        axes[0].plot(cgrid, integrate_beta(beta_shared, r0))
        axes[1].plot(cgrid, integrate_beta(beta_meta, r0))
        axes[2].plot(cgrid, integrate_beta(beta_true, r0))

    axes[0].set_title("Shared-law")
    axes[1].set_title(f"Meta-law ({best_name})")
    axes[2].set_title("Regime-specific")
    for ax in axes:
        ax.set_xlabel("condition coordinate c")
    axes[0].set_ylabel("residual")
    fig.suptitle(f"{title_prefix}: {regime_id}", y=1.03)
    plt.tight_layout()
    plt.show()

plot_case(success_row["regime_id"], "Representative success case")
plot_case(failure_row["regime_id"], "Representative harder case")

## Final verdicts

In [ ]:
def verdict_block(df_block):
    if df_block["traj_rmse_meta"].mean() < df_block["traj_rmse_shared"].mean() and df_block["coef_rmse_meta"].mean() < df_block["coef_rmse_shared"].mean():
        return "meta-law robust"
    if df_block["traj_rmse_meta"].mean() < df_block["traj_rmse_shared"].mean():
        return "meta-law partly robust"
    return "boundary of universality reached"

verdicts = pd.DataFrame([
    {"block": "k_extrapolation_interpolation", "verdict": verdict_block(k_holdout_df)},
    {"block": "forcing_family_holdout", "verdict": verdict_block(forcing_holdout_df)},
    {"block": "system_holdout", "verdict": verdict_block(system_holdout_df)},
    {"block": "task_holdout", "verdict": verdict_block(task_holdout_df)},
    {"block": "missing_point_interpolation", "verdict": verdict_block(interp_holdout_df)},
    {"block": "mixed_metadata_extrapolation", "verdict": verdict_block(mixed_holdout_df)},
])

display(verdicts)

## Working conclusion

Notebook 50 tests the boundary between interpolation and extrapolation for the metadata-conditioned law.

A strong outcome is:
- mild k extrapolation still works
- missing-point interpolation works very well
- some family holdouts remain tractable
- mixed or system-level shifts expose the boundary of universality

If that pattern holds on your real exports, the next notebook should be:

**Notebook 51 — boundary of universality and failure modes**